# 🏥 CaaS-Q: MedQCNN as an Agentic AI Tool

This notebook demonstrates how **MedQCNN** integrates into the **CaaS-Q (Company as a Service — Quantum)** agentic AI network.

## The Big Picture

MedQCNN is not a standalone app — it's a **quantum diagnostic tool** that AI agents call. The flow is:

```
Doctor uploads MRI → Agent receives image → Agent calls MedQCNN tool
  → Quantum inference runs → Agent gets ⟨σ_z⟩ probabilities
  → Agent writes clinical report → Doctor receives diagnosis
```

This notebook covers:
1. **MedQCNN as a LangChain Tool** — how the agent calls quantum inference
2. **Full Diagnostic Flow** — image → quantum analysis → clinical report
3. **LangChain Agent** — constructing a ReAct agent with MedQCNN tools

---

## 1. The Three Tools

MedQCNN exposes 3 tools that any LangChain agent can use:

| Tool | What It Does | CaaS-Q Role |
|------|-------------|-------------|
| `quantum_diagnose` | Image → quantum inference → prediction | Core diagnostic engine |
| `get_model_info` | Returns architecture details | Agent self-awareness |
| `list_medical_datasets` | Available benchmark datasets | Data catalog |

In [ ]:
import json
from medqcnn.agent.tools import quantum_diagnose, get_model_info, list_medical_datasets

# Tool 1: Model Info — the agent learns about its quantum capabilities
info = json.loads(get_model_info.invoke(""))
print("Model Architecture")
print("=" * 50)
print(json.dumps(info, indent=2))

In [ ]:
# Tool 2: List Datasets — the agent knows what data is available
datasets = json.loads(list_medical_datasets.invoke(""))
print("Available Medical Datasets")
print("=" * 50)
for name, ds in datasets.items():
    print(f"  {name:15s} — {ds['description']} ({ds['n_classes']} classes)")

## 2. Preparing a Test Image

Let's get a real medical image from BreastMNIST, save it as a file, and then run quantum diagnosis on it — exactly as an agent would.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from medqcnn.data.loader import get_medmnist_loaders
from PIL import Image
from pathlib import Path

# Load BreastMNIST
_, _, test_loader = get_medmnist_loaders('breastmnist', batch_size=1, download=True)

# Get one sample and save as PNG
images, labels = next(iter(test_loader))
img_array = (images[0].squeeze().numpy() * 255).astype(np.uint8)
save_path = Path('data/test_sample.png')
save_path.parent.mkdir(exist_ok=True)
Image.fromarray(img_array).save(save_path)

label_name = 'Malignant' if labels[0].item() == 1 else 'Benign'
print(f"Saved test image: {save_path}")
print(f"True label: {label_name} ({labels[0].item()})")

plt.figure(figsize=(4, 4))
plt.imshow(img_array, cmap='gray')
plt.title(f"Breast Ultrasound — {label_name}")
plt.axis('off')
plt.show()

## 3. Running Quantum Diagnosis (The Core Tool Call)

This is the exact call an AI agent makes. The tool:
1. Loads the image
2. Passes through frozen ResNet → projector → L2 norm
3. Runs the 4-qubit quantum circuit (amplitude encoding → HEA → Pauli-Z)
4. Returns prediction, confidence, and raw quantum expectation values

In [ ]:
# Tool 3: quantum_diagnose — this is what the agent calls
result_json = quantum_diagnose.invoke(str(save_path.resolve()))
result = json.loads(result_json)

print("Quantum Diagnosis Result")
print("=" * 50)
print(json.dumps(result, indent=2))

In [ ]:
# Visualize the quantum expectation values
q_values = result['quantum_expectation_values']
n_qubits = len(q_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of expectation values
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in q_values]
axes[0].bar(range(n_qubits), q_values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_xlabel('Qubit Index')
axes[0].set_ylabel('⟨σ_z⟩ Expectation Value')
axes[0].set_title('Per-Qubit Quantum Measurements')
axes[0].set_ylim(-1.1, 1.1)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xticks(range(n_qubits))
axes[0].set_xticklabels([f'Q{i}' for i in range(n_qubits)])
axes[0].grid(True, alpha=0.2)

# Probability pie chart
probs = result['probabilities']
axes[1].pie(
    [probs['Benign'], probs['Malignant']],
    labels=['Benign', 'Malignant'],
    colors=['#2ecc71', '#e74c3c'],
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 12},
)
axes[1].set_title(f"Classification: {result['prediction']} ({result['confidence']*100:.1f}% confidence)")

plt.suptitle('MedQCNN Quantum Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Full CaaS-Q Flow: Generating a Clinical Report

In the real CaaS-Q system, an LLM agent would take the quantum inference results and write a clinical report. Here we demonstrate this using the built-in report generator (no LLM API key required).

In [ ]:
from medqcnn.agent.agent import run_diagnostic_without_llm

# Generate a clinical report from the quantum diagnosis
report = run_diagnostic_without_llm(str(save_path.resolve()))
print(report)

## 5. Using with a Real LLM (Optional)

If you have an OpenAI or other LLM API key, you can create a full ReAct agent:

```python
from langchain_openai import ChatOpenAI
from medqcnn.agent.agent import create_agent_executor
from langchain_core.messages import HumanMessage

# Build agent with LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent = create_agent_executor(llm)

# Run diagnostic
result = agent.invoke({"messages": [
    HumanMessage(content="Analyze this breast ultrasound: /path/to/image.png")
]})

# The agent will:
# 1. Call quantum_diagnose with the image path
# 2. Interpret quantum expectation values
# 3. Write a clinical report
print(result['messages'][-1].content)
```

The agent can also be connected via **MCP** or **Kafka** for production deployment.

## 6. Integration Options

MedQCNN can be integrated into your AI platform in 3 ways:

| Method | Protocol | Best For |
|--------|----------|----------|
| **LangChain Tool** | Python import | Direct agent integration |
| **MCP Server** | stdio/SSE | Claude, Copilot, IDE agents |
| **REST API** | HTTP | Microservice / Docker |
| **Kafka** | Event-driven | Async high-throughput pipeline |

### MCP Configuration
```json
{
  "mcpServers": {
    "medqcnn": {
      "command": "uv",
      "args": ["run", "python", "scripts/mcp_server.py"],
      "cwd": "/path/to/MedQCNN"
    }
  }
}
```

### REST API
```bash
uv run python scripts/serve.py
curl -X POST http://localhost:8000/predict -d '{"image_base64": "..."}'
```

### Docker
```bash
docker compose up -d
```

---

## Summary

MedQCNN is a **quantum diagnostic endpoint** for the CaaS-Q agentic AI platform:

- 🧠 **4–8 qubit** quantum circuit with only 32–64 trainable parameters
- 🔬 **Amplitude encoding** maps clinical images into quantum Hilbert space
- 🤖 **3 integration methods**: LangChain tools, MCP server, REST API
- 📊 **Per-qubit measurements** provide interpretable quantum features
- ⚕️ **Clinical reports** generated by LLM agents from quantum inference results

The quantum advantage: 32 parameters operating in a 16-dimensional Hilbert space — vs thousands in classical FC layers.